Khai báo thư viện

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import haversine_distances
from math import radians

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from statsmodels.api import WLS
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

Load data

In [3]:
folder_data = r"D:\Download\Tai_lieu_hoc\BI\Final Project\Data"

df_buyer = pd.read_excel(folder_data + r"/Buyer.xlsx", sheet_name= "Buyer")
df_sellers = pd.read_excel(folder_data + r"/Seller.xlsx", sheet_name= "Seller")

df_geolocation = pd.read_excel(folder_data + r"/Geolocation.xlsx", sheet_name= "Geolocation")

df_orders = pd.read_excel(folder_data + r"/Order.xlsx", sheet_name= "Order")
df_order_items = pd.read_excel(folder_data + r"/Order Item.xlsx", sheet_name= "Order item")
df_products = pd.read_excel(folder_data + r"/Product.xlsx", sheet_name= "Product")

In [4]:
# 2. Gộp các bảng liên quan đến luồng đơn hàng (Core Flow)
df_geo_clean = df_geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean'
}).reset_index()
# Sau đó mới dùng df_geo_clean để merge
master_df = pd.merge(df_orders, df_buyer, on='buyer_id', how='left')
master_df = master_df.merge(df_geo_clean, left_on='buyer_zip_code_prefix',
                            right_on='geolocation_zip_code_prefix', how='left')
master_df.rename(columns={'geolocation_lat': 'buyer_lat', 'geolocation_lng': 'buyer_lng'}, inplace=True)
master_df.drop(columns=['geolocation_zip_code_prefix'], inplace=True)

# Gộp với Order Items (Lưu ý: 1 đơn có thể có nhiều items)
master_df = pd.merge(df_order_items, master_df, on='order_id', how='left')

# Gộp với Products để lấy thông tin vật lý sản phẩm
master_df = pd.merge(master_df, df_products, on='product_id', how='left')

# Gộp với Sellers để biết thông tin người bán
master_df = pd.merge(master_df, df_sellers, on='seller_id', how='left')
master_df = master_df.merge(df_geo_clean, left_on='seller_zip_code_prefix',
                            right_on='geolocation_zip_code_prefix', how='left')
master_df.rename(columns={'geolocation_lat': 'seller_lat', 'geolocation_lng': 'seller_lng'}, inplace=True)
master_df.drop(columns=['geolocation_zip_code_prefix'], inplace=True)

# 5. Xử lý định dạng thời gian cho toàn bộ bảng
time_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_buyer_date',
    'order_estimated_delivery_date', 'shipping_limit_date',
]

for col in time_columns:
    if col in master_df.columns:
        master_df[col] = pd.to_datetime(master_df[col])

# Kiểm tra kết quả
print(f"Tổng số cột: {len(master_df.columns)}")
print(f"Tổng số dòng: {len(master_df)}")
master_df.head()

Tổng số cột: 33
Tổng số dòng: 112646


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,sale_value,freight_value,buyer_id,order_status,order_purchase_timestamp,...,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng
0,e5fa5a7210941f7d56d0208e4e071d35,1,f3c2d01a84c947b078e32bbef0718962,a425f92c199eb576938df686728acd20,2023-09-19 00:15:34,59.50,15.56,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2023-09-05 00:15:34,...,1.0,700.0,25.0,2.0,25.0,81050.0,curitiba,PR,-25.495038,-49.299601
1,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2023-09-19 23:11:33,44.99,2.83,86dc2ffce2dfff336de2f386a786e574,delivered,2023-09-15 12:16:38,...,1.0,1000.0,16.0,16.0,16.0,81810.0,curitiba,PR,-25.507144,-49.272075
2,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2023-09-19 23:11:33,44.99,2.83,86dc2ffce2dfff336de2f386a786e574,delivered,2023-09-15 12:16:38,...,1.0,1000.0,16.0,16.0,16.0,81810.0,curitiba,PR,-25.507144,-49.272075
3,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2023-09-19 23:11:33,44.99,2.83,86dc2ffce2dfff336de2f386a786e574,delivered,2023-09-15 12:16:38,...,1.0,1000.0,16.0,16.0,16.0,81810.0,curitiba,PR,-25.507144,-49.272075
4,cd3b8574c82b42fc8129f6d502690c3e,1,e2a1d45a73dc7f5a7f9236b043431b89,b499c00f28f4b7069ff6550af8c1348a,2023-10-08 10:34:01,29.99,10.96,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2023-10-03 22:31:31,...,2.0,9000.0,16.0,5.0,33.0,13481.0,limeira,SP,-22.599254,-47.379810


Lọc và xử lý các feature cần thiết

In [5]:
master_df = master_df[master_df['order_status'] =='delivered'].copy()
master_df

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,sale_value,freight_value,buyer_id,order_status,order_purchase_timestamp,...,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng
1,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2023-09-19 23:11:33,44.99,2.83,86dc2ffce2dfff336de2f386a786e574,delivered,2023-09-15 12:16:38,...,1.0,1000.0,16.0,16.0,16.0,81810.0,curitiba,PR,-25.507144,-49.272075
2,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2023-09-19 23:11:33,44.99,2.83,86dc2ffce2dfff336de2f386a786e574,delivered,2023-09-15 12:16:38,...,1.0,1000.0,16.0,16.0,16.0,81810.0,curitiba,PR,-25.507144,-49.272075
3,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2023-09-19 23:11:33,44.99,2.83,86dc2ffce2dfff336de2f386a786e574,delivered,2023-09-15 12:16:38,...,1.0,1000.0,16.0,16.0,16.0,81810.0,curitiba,PR,-25.507144,-49.272075
4,cd3b8574c82b42fc8129f6d502690c3e,1,e2a1d45a73dc7f5a7f9236b043431b89,b499c00f28f4b7069ff6550af8c1348a,2023-10-08 10:34:01,29.99,10.96,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2023-10-03 22:31:31,...,2.0,9000.0,16.0,5.0,33.0,13481.0,limeira,SP,-22.599254,-47.379810
5,c3d9e402b6a0fbe2a5f7fc5b41117c38,1,817e1c2d22418c36386406ccacfa53e8,624f4ece8da4aafb77699233d480f8ef,2023-10-08 10:45:33,189.00,48.45,5720a15d022c09d2634c71c80c8d4102,delivered,2023-10-04 10:16:04,...,1.0,2750.0,50.0,50.0,16.0,5138.0,sao paulo,SP,-23.485254,-46.733130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112641,cf5c8d9f52807cb2d2f0a0ff54c478da,6,a7bbff32c7321478b29f924301a1867d,dfc475d54e1b6dbeeb7d7d9bdaa63827,2025-09-12 13:24:27,16.90,8.99,e898b5ef24833b9cb9e2d4f00b937595,delivered,2025-08-24 13:04:05,...,2.0,300.0,25.0,6.0,15.0,81460.0,curitiba,PR,-25.549428,-49.324493
112642,59eaa904b3f0dbde2785ac1b27eccd18,1,61919b39651acb61ec24307ed8b9502d,f61c63d13f7cd800549d5acdd390ae72,2025-09-13 14:55:28,299.00,14.75,3e90b5882ce0e665b837de00a2a8625c,delivered,2025-08-20 10:19:46,...,1.0,250.0,17.0,4.0,16.0,18185.0,pilar do sul,SP,-23.821237,-47.717391
112643,1afe384f199748cff7a42c9902065560,1,4c2a4020fcd651812100ebbeac1b2753,610f72e407cdd7caaa2f8167b0163fd8,2025-09-14 02:09:37,599.99,29.18,df646960391593c3f41cd448d84800c7,delivered,2025-08-21 01:45:43,...,2.0,22300.0,45.0,40.0,55.0,1201.0,sao paulo,SP,-23.534357,-46.649356
112644,7cfdf7265c9572fc7b7cbd3b9cc438b7,2,17e18b0c88a853dd6de3e48a7cfa9d9a,cee48807215b30a12ca2ca10ffb5f250,2025-09-14 12:30:56,20.00,19.25,00791d8bb3acb245dc0b865656e18fff,delivered,2025-08-21 12:20:32,...,1.0,700.0,20.0,10.0,11.0,11431.0,guaruja,SP,-23.997778,-46.276764


In [6]:
master_df.isna().sum().sort_values(ascending=False)

seller_lat                       2252
seller_lng                       2252
seller_city                      2003
seller_zip_code_prefix           2003
seller_state                     2003
product_description_lenght       1635
product_photos_qty               1635
product_name_lenght              1635
buyer_lng                         432
buyer_lat                         432
buyer_zip_code_prefix             144
buyer_city                        144
buyer_unique_id                   144
buyer_state                       144
product_height_cm                 117
product_weight_g                  117
product_length_cm                 117
product_width_cm                  117
product_category_name              99
order_approved_at                  15
order_delivered_buyer_date          8
order_delivered_carrier_date        2
order_id                            0
order_estimated_delivery_date       0
order_purchase_timestamp            0
product_id                          0
order_item_i

Tính khoảng cách người mua và người bán

In [7]:
def calculate_distance(row):
    # Nếu thiếu tọa độ thì trả về NaN
    if pd.isna(row['buyer_lat']) or pd.isna(row['seller_lat']):
        return np.nan

    buyer = [radians(row['buyer_lat']), radians(row['buyer_lng'])]
    seller = [radians(row['seller_lat']), radians(row['seller_lng'])]

    # Tính khoảng cách theo KM (bán kính Trái Đất ~ 6371km)
    result = haversine_distances([buyer, seller]) * 6371
    return result[0][1]

master_df['distance_km'] = master_df.apply(calculate_distance, axis=1)

In [8]:
master_df[['buyer_lat','buyer_lng','seller_lat','seller_lng','distance_km']].head()

,buyer_lat,buyer_lng,seller_lat,seller_lng,distance_km
1,-20.581177,-47.858931,-25.507144,-49.272075,566.488927
2,-20.581177,-47.858931,-25.507144,-49.272075,566.488927
3,-20.581177,-47.858931,-25.507144,-49.272075,566.488927
4,-23.036952,-45.569976,-22.599254,-47.379810,191.771897
5,-1.400029,-48.429567,-23.485254,-46.733130,2462.565619


xử lý dữ liệu thời gian

In [9]:
# Tạo cột Tháng
master_df['order_purchase_month'] = master_df['order_purchase_timestamp'].dt.month

# Tạo thêm các cột hữu ích cho dự đoán thời gian giao hàng
master_df['order_purchase_day_of_week'] = master_df['order_purchase_timestamp'].dt.dayofweek # 0=Thứ 2, 6=Chủ nhật
master_df['is_weekend'] = master_df['order_purchase_day_of_week'].isin([5, 6]).astype(int) # 1 nếu là cuối tuần, 0 nếu không

master_df['order_purchase_hour'] = master_df['order_purchase_timestamp'].dt.hour # Giờ trong ngày (0-23)
master_df['is_working'] = master_df['order_purchase_hour'].isin([9, 10, 11, 12, 13, 14, 15, 16, 17]).astype(int) # 1 nếu là giờ làm việc, 0 nếu không

In [10]:
# 1. Thời gian giao hàng thực tế (Target variable) - tính bằng ngày
# Chú ý: Chỉ tính được cho các đơn đã giao (delivered)
master_df['actual_delivery_days'] = (master_df['order_delivered_buyer_date'] - master_df['order_approved_at']).dt.total_seconds() / 86400
# 2. Thời gian xử lý của người bán (Từ lúc đặt -> Giao cho đơn vị vận chuyển)
master_df['carrier_pickup_days'] = ((master_df['order_delivered_carrier_date'] - master_df['order_approved_at']).dt.total_seconds() / 86400).round(0)
master_df['approval_delays'] = ((master_df['order_approved_at'] - master_df['order_purchase_timestamp']).dt.total_seconds() / 36000).round(0)
# 3. Thời gian giao hàng dự kiến của hệ thống
master_df['estimated_delivery_days'] = (master_df['order_estimated_delivery_date'] - master_df['order_purchase_timestamp']).dt.total_seconds() / 86400

# 4. Thời gian giới hạn giao hàng (shipping limit)
master_df['shipping_limit_days'] = (master_df['shipping_limit_date'] - master_df['order_approved_at']).dt.total_seconds() / 86400
#master_df['is_shipping_limited'] = (master_df['actual_delivery_days'] < (master_df['shipping_limit_days'])).astype(int)

In [11]:
import holidays
from datetime import timedelta

# 1. Khởi tạo lịch lễ của Brazil (BR)
br_holidays = holidays.BR()

def check_near_holiday(row, window_days=7):
    # Lấy ngày đặt hàng
    purchase_date = row['order_purchase_timestamp'].date()

    # Kiểm tra trong vòng 7 ngày tới có ngày lễ nào không
    for i in range(1, window_days + 1):
        future_date = purchase_date + timedelta(days=i)
        if future_date in br_holidays:
            return 1
    return 0

# 3. Áp dụng vào DataFrame của bạn
# Lưu ý: Đảm bảo các cột thời gian đã được chuyển sang dạng datetime
master_df['has_holiday'] = master_df.apply(check_near_holiday, axis=1)

Tạo biến xem rằng cùng bang/ thành phố có ảnh hưởng

In [12]:
master_df['same_state'] = (master_df['buyer_state'] == master_df['seller_state']).astype(int)
master_df['same_city'] = (master_df['buyer_city'] == master_df['seller_city']).astype(int)

xử lý dữ kiện về kích thước đơn hàng --> thể tích

In [13]:
# Tính thể tích sản phẩm (cm3)
master_df['product_volume_cm3'] = (
    master_df['product_length_cm'] * master_df['product_height_cm'] * master_df['product_width_cm']
)

kiểm tra xem đơn hàng có bao nhiêu người bán

In [14]:
# Đếm số người bán duy nhất cho mỗi đơn hàng
seller_counts = master_df.groupby('order_id')['seller_id'].nunique().sort_values(ascending=False)

print("Top 5 đơn hàng có nhiều người bán nhất:")
print(seller_counts.head())

# Thống kê tỷ lệ
print("\nPhân bổ số lượng người bán trên mỗi đơn hàng:")
print(seller_counts.value_counts(normalize=True) * 100)

Top 5 đơn hàng có nhiều người bán nhất:
order_id
1c11d0f4353b31ac3417fbfa5f0f2a8a    5
cf5c8d9f52807cb2d2f0a0ff54c478da    5
91be51c856a90d7efe86cf9d082d6ae3    4
1d23106803c48c391366ff224513fb7f    4
8c2b13adf3f377c8f2b06b04321b0925    4
Name: seller_id, dtype: int64

Phân bổ số lượng người bán trên mỗi đơn hàng:
seller_id
1    98.678441
2     1.260404
3     0.055972
4     0.003110
5     0.002073
Name: proportion, dtype: float64


kiểm tra các missing value

In [15]:

def check_missing_data(df):
    # Tính số lượng giá trị thiếu trên mỗi cột
    missing_count = df.isnull().sum()

    # Tính tỉ lệ phần trăm (%) thiếu
    missing_percent = (df.isnull().sum() / len(df)) * 100

    # Gộp thành một DataFrame để dễ quan sát
    missing_report = pd.DataFrame({
        'Số lượng thiếu': missing_count,
        'Tỉ lệ (%)': missing_percent
    })

    # Sắp xếp theo tỉ lệ giảm dần và chỉ hiện các cột có thiếu dữ liệu
    missing_report = missing_report[missing_report['Số lượng thiếu'] > 0].sort_values(by='Tỉ lệ (%)', ascending=False)

    return missing_report

# Chạy thử hàm
missing_info = check_missing_data(master_df)
print(missing_info)  


                              Số lượng thiếu  Tỉ lệ (%)
distance_km                             2675   2.427515
seller_lat                              2252   2.043650
seller_lng                              2252   2.043650
seller_city                             2003   1.817687
seller_state                            2003   1.817687
seller_zip_code_prefix                  2003   1.817687
product_description_lenght              1635   1.483733
product_photos_qty                      1635   1.483733
product_name_lenght                     1635   1.483733
buyer_lng                                432   0.392032
buyer_lat                                432   0.392032
buyer_state                              144   0.130677
buyer_unique_id                          144   0.130677
buyer_zip_code_prefix                    144   0.130677
buyer_city                               144   0.130677
product_volume_cm3                       117   0.106175
product_width_cm                         117   0

In [16]:
# Danh sách các cột datetime ban đầu
cols_to_drop = [
    #id
    'buyer_id', 'buyer_unique_id', 'product_id',
    #dữ liệu thời gian
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_buyer_date',
    'order_estimated_delivery_date',  'shipping_limit_date','order_purchase_day_of_week','order_purchase_hour',
    #các cột liên quan đến địa lý
    'seller_city', 'buyer_city','buyer_lat', 'buyer_lng', 'seller_lat', 'seller_lng',
    #mã vùng
    'buyer_zip_code_prefix', 'seller_zip_code_prefix',
    #kích thước sản phẩm
    'product_length_cm', 'product_height_cm', 'product_width_cm',
    #nội dung và hình ảnh sản phẩm
    'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_category_name',
    #trạng thái giao hàng
    'order_status'
]

# Chỉ xóa những cột tồn tại trong DataFrame để tránh lỗi
existing_cols_to_drop = [c for c in cols_to_drop if c in master_df.columns]
df_final = master_df.drop(columns=existing_cols_to_drop).dropna().copy()


In [17]:
df_final.columns

Index(['order_id', 'order_item_id', 'seller_id', 'sale_value', 'freight_value',
       'buyer_state', 'product_weight_g', 'seller_state', 'distance_km',
       'order_purchase_month', 'is_weekend', 'is_working',
       'actual_delivery_days', 'carrier_pickup_days', 'approval_delays',
       'estimated_delivery_days', 'shipping_limit_days', 'has_holiday',
       'same_state', 'same_city', 'product_volume_cm3'],
      dtype='object')

encode dữ liệu

In [18]:
df_final.select_dtypes(include=['object']).info()

<class 'pandas.core.frame.DataFrame'>
Index: 107380 entries, 1 to 112645
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   order_id      107380 non-null  object
 1   seller_id     107380 non-null  object
 2   buyer_state   107380 non-null  object
 3   seller_state  107380 non-null  object
dtypes: object(4)
memory usage: 4.1+ MB


In [19]:
from category_encoders import TargetEncoder

# 1. Phân nhóm các cột
high_card_cols = ['buyer_state']

# 2. Xử lý Target Encoding (Thay tên bằng giá trị trung bình của biến mục tiêu)
# Điều này giúp mô hình hiểu được "mức độ trễ" trung bình của từng thành phố
te = TargetEncoder(cols=high_card_cols)
df_encode = te.fit_transform(df_final, df_final['actual_delivery_days'])

# 3. Kiểm tra kết quả
print("Danh sách các cột sau khi Encode:")
print(df_encode.columns.tolist())
print(f"\nKích thước DataFrame mới: {df_encode.shape}")


Danh sách các cột sau khi Encode:
['order_id', 'order_item_id', 'seller_id', 'sale_value', 'freight_value', 'buyer_state', 'product_weight_g', 'seller_state', 'distance_km', 'order_purchase_month', 'is_weekend', 'is_working', 'actual_delivery_days', 'carrier_pickup_days', 'approval_delays', 'estimated_delivery_days', 'shipping_limit_days', 'has_holiday', 'same_state', 'same_city', 'product_volume_cm3']

Kích thước DataFrame mới: (107380, 21)


In [54]:
df_final = df_encode.groupby('order_id').agg({
    'actual_delivery_days': 'first',  # Target variable
    'same_state': 'min', # đơn hàng có giao khác ban không?
    'same_city': 'min', # đơn hàng có giao khác thành phố không?
    'sale_value': 'sum',  # Tổng giá trị đơn hàng
    'freight_value': 'sum',  # Tổng giá trị vận chuyển
    'product_weight_g': 'sum',  # Tổng trọng lượng sản phẩm trong đơn hàng
    'product_volume_cm3': 'sum',  # Tổng thể tích sản phẩm trong đơn hàng
    'order_purchase_month': 'first',  # Lấy tháng diễn ra đơn hàng
    'carrier_pickup_days': 'max', # thời gian xử lý của người bán lâu nhất trong đơn hàng
    'approval_delays': 'max', # thời gian chậm trễ phê duyệt lâu nhất trong đơn hàng
    'shipping_limit_days': 'min', # thời gian giới hạn giao hàng ngắn nhất trong đơn hàng
#    'is_shipping_limited': 'min', # nếu có sản phẩm nào bị giao hàng trễ thì cả đơn hàng đó bị trễ
    'distance_km': 'max',
    'has_holiday': 'max', # kiểm tra xem có sản phẩm nào trong đơn hàng bị ảnh hưởng bởi ngày lễ không
    'is_weekend': 'max', # kiểm tra xem có sản phẩm nào trong đơn hàng được đặt vào cuối tuần không
    'is_working': 'min', # kiểm tra xem có sản phẩm nào trong đơn hàng được thanh toán ngoài giờ làm việc không
    'buyer_state': 'first', # Lấy giá trị đầu tiên (hoặc có thể là giá trị phổ biến nhất nếu có nhiều sản phẩm trong đơn)
#    'product_category_name': 'first',  # Lấy giá trị đầu tiên (hoặc có thể là giá trị phổ biến nhất nếu có nhiều sản phẩm trong đơn)
    'seller_id': 'nunique', # số lượng người bán trong đơn hàng
    'order_item_id': 'max' # số lượng sản phẩm trong đơn hàng
    }).copy()
df_final.rename(columns={'seller_id': 'num_sellers',
                         'order_item_id': 'num_products'
                         }, inplace=True)

In [55]:
# def calculate_weight(row):
#     delay = row['actual_delivery_days'] - row['shipping_limit_days']
#     if delay > 0:
#         return 1 + delay # Trễ càng nhiều, trọng số càng nặng
#     else:
#         return 1 # Đúng hạn thì coi như bình thường

# df_train_encode_group_id['weights'] = df_train_encode_group_id.apply(calculate_weight, axis=1)

In [56]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94339 entries, 00010242fe8c5a6d1ba2dd792cb16214 to fffe41c64501cc87c801fd61db3f6244
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   actual_delivery_days  94339 non-null  float64
 1   same_state            94339 non-null  int64  
 2   same_city             94339 non-null  int64  
 3   sale_value            94339 non-null  float64
 4   freight_value         94339 non-null  float64
 5   product_weight_g      94339 non-null  float64
 6   product_volume_cm3    94339 non-null  float64
 7   order_purchase_month  94339 non-null  int32  
 8   carrier_pickup_days   94339 non-null  float64
 9   approval_delays       94339 non-null  float64
 10  shipping_limit_days   94339 non-null  float64
 11  distance_km           94339 non-null  float64
 12  has_holiday           94339 non-null  int64  
 13  is_weekend            94339 non-null  int64  
 14  is_working       

In [57]:
import matplotlib.pyplot as plt
import seaborn as sns

In [59]:
df_final = df_final[(df_final['actual_delivery_days'] < 120)
                    & (df_final['sale_value'] < 4000)
                    & (df_final['freight_value'] < 500)
                    & (df_final['product_weight_g'] < 100000)
                    & (df_final['product_volume_cm3'] < 600000)
                    & (df_final['shipping_limit_days'] < 60)
                    & (df_final['distance_km'] < 4000)


                    ].copy()

Scaler

In [60]:
from sklearn.preprocessing import StandardScaler

# 1. Khởi tạo
scaler = StandardScaler()

Huấn luyện mô hình

In [61]:
df_final.shape

(94265, 18)

In [62]:
X = df_final.drop(columns=['actual_delivery_days']).copy()
y = df_final['actual_delivery_days'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [63]:

from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

param_grid = {
    'n_estimators': [100, 500, 1000],        # Số lượng cây
    'max_depth': [3, 6, 9],                  # Độ sâu của cây
    'learning_rate': [0.01, 0.1, 0.2],       # Tốc độ học
    'subsample': [0.7, 0.8, 1.0],            # Tỷ lệ dữ liệu dùng cho mỗi cây
    'colsample_bytree': [0.7, 0.8, 1.0]      # Tỷ lệ đặc trưng dùng cho mỗi cây
}

# 4. Thiết lập RandomizedSearchCV
# cv=5 nghĩa là thực hiện K-fold 5 lần để đảm bảo kết quả ổn định
grid_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    cv=5, 
    scoring='neg_mean_absolute_error', # Tối ưu hóa theo hướng giảm sai số ngày giao hàng
    n_jobs=-1 # Sử dụng toàn bộ nhân CPU để chạy nhanh hơn
)

# 5. Tiến hành Train và tìm tham số tốt nhất
print("Đang bắt đầu tìm kiếm tham số tối ưu... Vui lòng đợi.")
grid_search.fit(X_train_scaled, y_train)

# 6. Kết quả
print(f"Tham số tốt nhất: {grid_search.best_params_}")
best_model = grid_search.best_estimator_

n = len(y_test)
p = X_test.shape[1] 

# Dự báo và đánh giá sơ bộ
y_pred = best_model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"Mean Absolute Error (MAE) sau khi Tuned: {mae:.2f} ngày")
print(f"R2 Score: {r2:.4f}")
print(f"Adjusted R2: {adj_r2}")

Đang bắt đầu tìm kiếm tham số tối ưu... Vui lòng đợi.
Tham số tốt nhất: {'subsample': 1.0, 'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
Mean Absolute Error (MAE) sau khi Tuned: 3.94 ngày
R2 Score: 0.4895
Adjusted R2: 0.48903297288064884


In [ ]:
import pandas as pd
from sklearn.metrics import r2_score

def get_adj_r2(X_train, y, model):
    model.fit(X_train, y)
    y_pred = model.predict(X_train)
    r2 = r2_score(y, y_pred)
    n, p = X.shape
    if n <= p + 1: return -1 # Tránh lỗi chia cho 0
    return 1 - (1 - r2) * (n - 1) / (n - p - 1) , r2


In [ ]:
features_names = X_train.columns 
X_df = pd.DataFrame(X_train_scaled, columns=features_names)
xgb_model=XGBRegressor(**grid_search.best_params_, objective='reg:squarederror', random_state=42)
remaining_features = list(X_df.columns)
selected_features = []
current_adj_r2 = -float('inf')

print(f"{'Biến thêm vào':<25} | {'Adj R2 mới':<12} | {'R2 mới':<12} | {'Cải thiện'}")
print("-" * 70)

while remaining_features:
    best_feature = None
    best_new_adj_r2 = -float('inf')
    
    for feature in remaining_features:
        # Thử thêm từng biến vào danh sách đã chọn
        features_to_test = selected_features + [feature]
        adj_r2, r2 = get_adj_r2(X_df[features_to_test], y_train, xgb_model)
        
        if adj_r2 > best_new_adj_r2:
            best_new_adj_r2 = adj_r2
            r2_for_best = r2
            best_feature = feature
            
    # Điều kiện dừng: Chỉ thêm nếu Adj R2 tăng "đáng kể" (ví dụ > 0.01)
    if best_new_adj_r2 > current_adj_r2 + 0.01:
        improvement = best_new_adj_r2 - current_adj_r2 if current_adj_r2 != -float('inf') else best_new_adj_r2
        remaining_features.remove(best_feature)
        selected_features.append(best_feature)
        current_adj_r2 = best_new_adj_r2
        print(f"{best_feature:<25} | {current_adj_r2:.4f}       | {r2_for_best:.4f}       | +{improvement:.4f}")
    else:
        print("\n--- Dừng lại vì Adj R2 không tăng đáng kể nữa ---")
        mae = mean_absolute_error(y_test, y_pred_final)
        r2 = r2_score(y_test, y_pred_final)
        rmse = root_mean_squared_error(y_test, y_pred_final)

        break

print(f"\nDanh sách biến tối ưu cuối cùng ({len(selected_features)} biến):")
print(selected_features)


Biến thêm vào             | Adj R2 mới   | R2 mới       | Cải thiện
----------------------------------------------------------------------


distance_km               | 0.1999       | 0.1999       | +0.1999
carrier_pickup_days       | 0.3664       | 0.3664       | +0.1665
order_purchase_month      | 0.4423       | 0.4423       | +0.0759
buyer_state               | 0.5029       | 0.5029       | +0.0606
freight_value             | 0.5292       | 0.5292       | +0.0263

--- Dừng lại vì Adj R2 không tăng đáng kể nữa ---

Danh sách biến tối ưu cuối cùng (5 biến):
['distance_km', 'carrier_pickup_days', 'order_purchase_month', 'buyer_state', 'freight_value']


In [ ]:
X_train_selected = X_df[selected_features].copy()

param_grid = {
    'n_estimators': [100, 500, 1000],        # Số lượng cây
    'max_depth': [3, 6, 9],                  # Độ sâu của cây
    'learning_rate': [0.01, 0.1, 0.2],       # Tốc độ học
    'subsample': [0.7, 0.8, 1.0],            # Tỷ lệ dữ liệu dùng cho mỗi cây
    'colsample_bytree': [0.7, 0.8, 1.0]      # Tỷ lệ đặc trưng dùng cho mỗi cây
}

grid_search_final = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)
print("Đang bắt đầu tìm kiếm tham số tối ưu trên tập biến đã chọn... Vui lòng đợi.")
grid_search_final.fit(X_train_selected, y_train)

print(f"Tham số tốt nhất sau khi chọn biến: {grid_search_final.best_params_}")
best_model_final = grid_search_final.best_estimator_

X_test_selected = pd.DataFrame(X_test_scaled, columns=features_names)[selected_features].copy()
y_pred_final = best_model_final.predict(X_test_selected)
mae = mean_absolute_error(y_test, y_pred_final)
r2 = r2_score(y_test, y_pred_final)
rmse = root_mean_squared_error(y_test, y_pred_final)


Đang bắt đầu tìm kiếm tham số tối ưu trên tập biến đã chọn... Vui lòng đợi.
Tham số tốt nhất sau khi chọn biến: {'subsample': 0.7, 'n_estimators': 500, 'max_depth': 9, 'learning_rate': 0.01, 'colsample_bytree': 0.8}


In [76]:
mae = mean_absolute_error(y_test, y_pred_final)
r2 = r2_score(y_test, y_pred_final)
rmse = root_mean_squared_error(y_test, y_pred_final)
print(f'MAE  : {mae:.4f}')
print(f'R2   : {r2:.4f}')
print(f'RMSE : {rmse:.4f}')

MAE  : 3.9901
R2   : 0.4839
RMSE : 6.3658
